In [12]:
import pickle 
dataset_dir = "datasets/nations/blockpositionindex/"
with open(dataset_dir + 'x_train.dt', 'rb') as f:
    x_train = pickle.load(f)
    f.close()

with open(dataset_dir + 'y_train.dt', 'rb') as f:
    y_train = pickle.load(f)
    f.close()

print(f"x_train length: {len(x_train[0])}")
print(f"y_train length: {len(y_train)}")

x_train length: 38416
y_train length: 38416


In [13]:
import numpy as np

# Find the shape of the input data

num_samples = len(x_train)
num_features = len(x_train[0]) if num_samples > 0 else 0
print("Number of samples in x_train:", num_samples)
print("Number of features in each sample:", num_features)

Number of samples in x_train: 56
Number of features in each sample: 38416


In [16]:
import torch
x_train_np = np.concatenate(x_train, axis=1)
X_train = torch.tensor(x_train_np[:, :-1], dtype=torch.float32)
y_train = torch.tensor(x_train_np[:, -1], dtype=torch.float32)

# Move to device
# X_train = X_train.to(device)
# y_train = y_train.to(device)
from utils import check_binary_dataset, create_V_matrix, extract_head_and_connected_vars
from main import get_predicate_name
file_path = dataset_dir
synthetic_data = False
structured_data = True

if check_binary_dataset(X_train):
    dataset_type = "binary"
    print("X_train is a binary dataset. We are perturbing the data for a better training.")
    # perturb data by adding small noise to zero and reducing one to less than one
    X_train = X_train - 0.1 * torch.rand_like(X_train) * (X_train - 0.5).sign()
    # clip to [0, 1]
    X_train = torch.clamp(X_train, 0.0, 1.0)

# Load the predicate list
with open(file_path + 'predicate.list', 'rb') as f:
    predicate_list = pickle.load(f)

# Load relation variable permutations
with open(file_path + 'relation_variable.dt', 'rb') as f:
    relation_variable_permutations = pickle.load(f)

# Load relation names
with open(file_path + 'all_relation_dic.dt', 'rb') as f:
    all_relations = pickle.load(f)

print("Predicates:", len(predicate_list))
print("Relations:", len(all_relations))
print("Variable permutations:", len(relation_variable_permutations))

# Example: Map X_train column 0
body_predicates = []
for idx in range(len(predicate_list)):
    predicate_name = get_predicate_name(idx, predicate_list, 
                                        relation_variable_permutations, 
                                        all_relations)
    body_predicates.append(predicate_name)

# check if the length of body_predicates matches the number of columns in X_train
assert len(body_predicates) == X_train.shape[1], "Mismatch between body predicates and X_train columns."
_, variables = create_V_matrix(body_predicates)
head_predicate = "blockpositionindex(X1, X2)"  # Example head predicate
head_vars, connected_vars = extract_head_and_connected_vars(head_predicate, variables)

# head_predicate = "blockpositionindex(X1, X2)" #body_predicates[252]  # Example head predicate
for idx in range(len(predicate_list)):
    if body_predicates[idx] == head_predicate:
        headpred_idx = idx
        print(f"Found head predicate at index {idx} in body predicates.")
        break

# remove the head predicate from body predicates
body_predicates.remove(head_predicate)
print("Number of body predicates:", len(body_predicates))

# create variable matrix without head predicate
variable_matrix, _ = create_V_matrix(body_predicates)

# remove the corresponding column from X_train
X_train = torch.cat((X_train[:, :headpred_idx], X_train[:, headpred_idx+1:]), dim=1)


# Remove predicates containing 'X4' (only for non-synthetic structured data)
if not synthetic_data and structured_data:
    indices_keep = [i for i, p in enumerate(body_predicates) if "X4" not in p]
    indices_remove = [i for i, p in enumerate(body_predicates) if "X4" in p]

    if not indices_remove:
        print("No predicates containing 'X4' were found. No changes made.")
    else:
        # Update body_predicates
        body_predicates = [body_predicates[i] for i in indices_keep]
        variable_matrix, variables = create_V_matrix(body_predicates)
        head_vars, connected_vars = extract_head_and_connected_vars(head_predicate, variables)
        # Update X_train (keep columns corresponding to indices_keep)
        X_train = X_train[:, indices_keep].clone()
        # Update input_dim
        input_dim = X_train.shape[1]
        # Summary
        print(f"Removed {len(indices_remove)} predicates containing 'X4'.")
        print(f"Old input_dim -> New input_dim: {input_dim}")

X_train is a binary dataset. We are perturbing the data for a better training.
Predicates: 660
Relations: 55
Variable permutations: 55
Found head predicate at index 252 in body predicates.
Number of body predicates: 659
Removed 330 predicates containing 'X4'.
Old input_dim -> New input_dim: 329


In [17]:
X_train.shape, y_train.shape

(torch.Size([38416, 329]), torch.Size([38416]))

In [28]:
from sklearn.model_selection import train_test_split

# Split the data into train and test sets (e.g., 80% train, 20% test)
X_tr, X_te, y_tr, y_te = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

print("Train set shape:", X_tr.shape, y_tr.shape)
print("Test set shape:", X_te.shape, y_te.shape)

Train set shape: torch.Size([30732, 329]) torch.Size([30732])
Test set shape: torch.Size([7684, 329]) torch.Size([7684])


In [29]:
# Split the test data into positive and negative samples based on y_te
X_te_pos = X_te[y_te == 1]
X_te_neg = X_te[y_te == 0]

print("Positive test samples:", X_te_pos.shape[0])
print("Negative test samples:", X_te_neg.shape[0])

Positive test samples: 2039
Negative test samples: 5645


In [35]:
print("X_te_pos shape:", X_te_pos.shape)
print("X_te_neg shape:", X_te_neg.shape)

def top_k_cosine_similar(X_pos, X_neg, k=5):
    dot_products = torch.mm(X_pos, X_neg.t())
    pos_norms = torch.norm(X_pos, p=2, dim=1, keepdim=True)
    neg_norms = torch.norm(X_neg, p=2, dim=1, keepdim=True).t()
    similarity_matrix = dot_products / (pos_norms * neg_norms + 1e-8)
    topk_values, topk_indices = torch.topk(similarity_matrix, k, dim=1)
    return topk_indices, topk_values

corrupted_indices, corrupted_values = top_k_cosine_similar(X_te_pos, X_te_neg, k=100)
print("Top-k indices:\n", corrupted_indices)
print("Top-k cosine similarities:\n", corrupted_values)


X_te_pos shape: torch.Size([2039, 329])
X_te_neg shape: torch.Size([5645, 329])
Top-k indices:
 tensor([[2871, 2605, 2609,  ..., 3209, 5273,  922],
        [1830, 3753,  357,  ...,  591, 1014, 2855],
        [1524, 4703, 1934,  ..., 3477, 2677, 5038],
        ...,
        [2592, 3257,  107,  ..., 4665, 2496, 4118],
        [1515, 3473, 4078,  ..., 1648, 3768, 1893],
        [3226,  791,  984,  ...,  834, 1401, 1944]])
Top-k cosine similarities:
 tensor([[0.7927, 0.7913, 0.7897,  ..., 0.6271, 0.6269, 0.6258],
        [0.7401, 0.7391, 0.7313,  ..., 0.5926, 0.5924, 0.5918],
        [0.8532, 0.7432, 0.7431,  ..., 0.5588, 0.5581, 0.5575],
        ...,
        [0.7529, 0.7527, 0.7473,  ..., 0.5574, 0.5573, 0.5566],
        [0.7841, 0.7785, 0.7771,  ..., 0.6016, 0.6016, 0.6014],
        [0.7826, 0.7810, 0.7784,  ..., 0.5826, 0.5807, 0.5804]])


In [31]:
# Load the ANDRE model for scoring
from ANDRE_Net2 import ANDRE
import os

# Specify the directory where the trained model is saved
# Update this path to the actual best model directory if needed
andre_model_dir = dataset_dir + "saved_models/andre_best_model_2025-12-17"  # Example path
andre_model_name = "ANDRE_best_model_12-48-55"  # Default name used in save_model

# Load the model (loads both weights and config)
andre_model = ANDRE.load_model(andre_model_dir, model_name=andre_model_name)

# Set model to evaluation mode
andre_model.eval()

# Now andre_model is ready to be used for scoring samples, e.g.:
# scores, _, _, _ = andre_model(X)
# where X is a torch tensor of shape [N, D]

Model loaded successfully from datasets/nations/blockpositionindex/saved_models/andre_best_model_2025-12-17
  - Weights: datasets/nations/blockpositionindex/saved_models/andre_best_model_2025-12-17/ANDRE_best_model_12-48-55_weights.pt
  - Config:  datasets/nations/blockpositionindex/saved_models/andre_best_model_2025-12-17/ANDRE_best_model_12-48-55_config.json
  - Num Rules: 1, Input Dim: 329


ANDRE(
  (rules): ModuleList(
    (0): RuleBlock(
      (activation): ProbSumActivation()
    )
  )
  (criterion): BCELoss()
)

In [36]:
import random

# Concatenate each X_te_pos with its top-k similar negative samples
# and compute the rank of the positive sample using a random scoring model


ranks = []
for i in range(X_te_pos.shape[0]):
    # Get the positive sample
    pos_sample = X_te_pos[i].unsqueeze(0)
    # Get the top-k negative samples for this positive sample
    neg_indices = corrupted_indices[i]
    neg_samples = X_te_neg[neg_indices]
    # Concatenate positive and negative samples
    all_samples = torch.cat([pos_sample, neg_samples], dim=0)
    if i == 0:
        print("All samples shape:", all_samples.shape)
    # Generate random scores for all samples
    # scores = torch.tensor([random.random() for _ in range(all_samples.shape[0])])
    # scores = torch.tensor([float(random.random() > 0.5) for _ in range(all_samples.shape[0])])
    # generate scores randomly from {0, 0.1, 0.2, ..., 1.0}
    # scores = torch.tensor([round(random.uniform(0, 1), 1) for _ in range(all_samples.shape[0])])
    # Match all_samples with the ANDRE model
    # all_samples = all_samples.reshape()
    # Use the ANDRE model to score all samples
    with torch.no_grad():
        scores, _, _, _ = andre_model(all_samples)

    # round scores to two decimal place
    scores = torch.round(scores * 10) / 10

    # From the scores, only keep the first score from the duplicate scores
    # if the score is [1, 0, 1, 1, 0], we keep [1, 0]
    new_scores = []
    seen = set()
    for score in scores:
        if score.item() not in seen:
            new_scores.append(score)
            seen.add(score.item())
    new_scores = torch.tensor(new_scores)

    # Get the rank of the positive sample (index 0 after concatenation)
    sorted_indices = torch.argsort(new_scores, descending=True)
    rank = (sorted_indices == 0).nonzero(as_tuple=True)[0].item() + 1  # ranks start from 1
    ranks.append(rank)

    if i == 0:
        print("Scores:", scores)
        print("New Scores:", new_scores)
        print("Sorted indices:", sorted_indices)
        print("Rank of the positive sample:", rank)

print("Ranks of positive samples:", len(ranks))

All samples shape: torch.Size([101, 329])
Scores: tensor([1.0000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 1.0000, 1.0000,
        1.0000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000,
        0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.0000,
        0.1000, 0.1000, 0.1000, 0.1000, 0.0000, 0.1000, 0.1000, 0.1000, 0.1000,
        0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000,
        0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000,
        0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000,
        0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000,
        0.1000, 0.1000, 0.1000, 1.0000, 1.0000, 0.1000, 0.1000, 0.1000, 0.1000,
        0.1000, 0.1000, 1.0000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000,
        0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000,
        0.1000, 1.0000])
New Scores: tensor([1.0000, 0.1000, 0.0000])


In [37]:
def mean_reciprocal_rank(ranks):
    """Compute Mean Reciprocal Rank (MRR) from a list of ranks."""
    reciprocal_ranks = [1.0 / r for r in ranks]
    return sum(reciprocal_ranks) / len(reciprocal_ranks)

def hits_at_k(ranks, k):
    """Compute Hits@K from a list of ranks."""
    hits = sum(1 for r in ranks if r <= k)
    return hits / len(ranks)

# Compute MRR
mrr = mean_reciprocal_rank(ranks)
print(f"Mean Reciprocal Rank (MRR): {mrr:.4f}")

# Compute Hits@K for K = 1, 3, 5, 10
for k in [1, 3, 5, 10]:
    hits = hits_at_k(ranks, k)
    print(f"Hits@{k}: {hits:.4f}")

Mean Reciprocal Rank (MRR): 0.9596
Hits@1: 0.9196
Hits@3: 1.0000
Hits@5: 1.0000
Hits@10: 1.0000
